# Phase 1 — Data Pipeline & Evaluation Metrics
**Project:** Can LoRA-Adapted SAM Match Specialist Polyp Networks?

Deliverables:
1. Install dependencies
2. Download all five polyp datasets (PraNet standard packages)
3. Verify train/seen/unseen splits
4. Visualise sample images from each dataset
5. Smoke-test all six evaluation metrics

**GPU needed:** None — T4 or CPU both fine for this notebook.

## 1. Setup

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules
print(f'Running in Colab: {IN_COLAB}')

if IN_COLAB:
    REPO = '/content/msu2026summer_final_project'
    if not os.path.exists(REPO):
        !git clone --quiet https://github.com/palism1/msu2026summer_final_project.git {REPO}
    %cd {REPO}
    # Hard-reset to latest remote to fix any stale local state
    !git fetch --quiet origin && git reset --hard origin/main
    # Clear Python bytecode cache so stale .pyc files cannot shadow updated .py files
    !find . -name '__pycache__' -type d -exec rm -rf {} + 2>/dev/null; true
    !pip install -q -r requirements.txt
else:
    print('Running locally — make sure you are in the repo root.')

repo_root = os.getcwd()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

DATA_ROOT = Path('data/polyp')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Data root: {DATA_ROOT.resolve()}')

## 2. Download Datasets

PraNet ships two Google Drive packages. Both download automatically below.

| Package | Size | Contents |
|---|---|---|
| TrainDataset | 399.5 MB | 1,450 images flat in `image/` + `masks/` |
| TestDataset  | 327.2 MB | 5 subfolders: Kvasir, CVC-ClinicDB, CVC-ColonDB, ETIS-LaribPolypDB, CVC-300 |

In [ ]:
import gdown, shutil, zipfile

TRAIN_ID = '1YiGHLw4iTvKdvbT6MgwO9zcCv8zJ_Bnb'
TEST_ID  = '1Y2z7FD5p5y31vkZwQQomXFRB0HutHyao'

def download_and_extract(file_id, zip_name):
    folder_name = zip_name.replace('.zip', '')
    target = DATA_ROOT / folder_name
    zip_path = f'/tmp/{zip_name}'

    if target.exists():
        n = len(list(target.rglob('*.jpg')) + list(target.rglob('*.png')))
        print(f'{folder_name}: already present ({n} files)')
        return

    print(f'Downloading {folder_name}...')
    gdown.download(id=file_id, output=zip_path, quiet=False, fuzzy=True)

    print('Extracting...')
    tmp = Path(f'/tmp/extract_{folder_name}')
    tmp.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(tmp)

    extracted = [p for p in tmp.iterdir() if not p.name.startswith('__')]
    if len(extracted) == 1 and extracted[0].is_dir():
        shutil.move(str(extracted[0]), str(target))
    else:
        target.mkdir(exist_ok=True)
        for item in extracted:
            shutil.move(str(item), str(target / item.name))
    print(f'Done -> {target}')

download_and_extract(TRAIN_ID, 'TrainDataset.zip')
download_and_extract(TEST_ID,  'TestDataset.zip')

## 3. Verify Folder Structure

**Note on actual layout from PraNet zips:**
- `TrainDataset/image/` — all 1,450 training images flat (no per-dataset subfolders)
- `TrainDataset/masks/` — corresponding masks
- `TestDataset/TestDataset/{dataset}/images/` — the zip adds a nested folder with the same name

In [ ]:
from src.data.dataset import _find_test_root

TEST_ROOT = _find_test_root(DATA_ROOT)
print(f'Test root resolved to: {TEST_ROOT}')

# Actual paths to check
checks = {
    'TrainDataset/image  (train)':            (DATA_ROOT / 'TrainDataset' / 'image',           1450),
    'TrainDataset/masks  (train)':            (DATA_ROOT / 'TrainDataset' / 'masks',            1450),
    'TestDataset/.../Kvasir/images':          (TEST_ROOT / 'Kvasir'           / 'images',        100),
    'TestDataset/.../CVC-ClinicDB/images':    (TEST_ROOT / 'CVC-ClinicDB'     / 'images',         62),
    'TestDataset/.../CVC-ColonDB/images':     (TEST_ROOT / 'CVC-ColonDB'      / 'images',        380),
    'TestDataset/.../ETIS-LaribPolypDB/img':  (TEST_ROOT / 'ETIS-LaribPolypDB'/ 'images',        196),
    'TestDataset/.../CVC-300/images':         (TEST_ROOT / 'CVC-300'          / 'images',         60),
}

print(f'\n{"Path":<42} {"Expected":>10} {"Found":>10} {"OK":>5}')
print('-' * 72)
all_ok = True
for label, (folder, expected) in checks.items():
    found = len(list(folder.glob('*'))) if folder.exists() else 0
    ok = '✓' if found == expected else '✗'
    if ok == '✗': all_ok = False
    print(f'{label:<42} {expected:>10} {found:>10} {ok:>5}')
print()
print('All good!' if all_ok else 'Something is off — see rows marked ✗')

## 4. Verify PraNet Splits

In [ ]:
from src.data import build_splits
from src.data.dataset import verify_splits

splits = build_splits(DATA_ROOT, seed=42)
verify_splits(splits)

In [ ]:
# Content-based overlap check (MD5 hash, not filename).
# Filename comparison gives false positives: each dataset numbers files
# independently (Kvasir has 1.jpg, CVC-ColonDB also has 1.jpg — completely
# different images from different clinics). We compare actual file contents.
import hashlib

def md5(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(65536), b''):
            h.update(block)
    return h.hexdigest()

print('Hashing 1,450 training images...')
train_hashes = {md5(p) for p in splits['train']['image_paths']}

for key in ['seen_kvasir', 'seen_clinicdb', 'cvc_colondb', 'etis_larib', 'cvc_300']:
    test_hashes = {md5(p) for p in splits[key]['image_paths']}
    overlap = train_hashes & test_hashes
    status = '✓ No content overlap' if not overlap else f'✗ IDENTICAL IMAGES: {len(overlap)}'
    print(f'{key:<20}: {status}')

## 5. Visualise Samples

In [ ]:
def show_samples(image_paths, mask_paths, title, n=4):
    n = min(n, len(image_paths))
    fig, axes = plt.subplots(2, n, figsize=(4*n, 5))
    fig.suptitle(title, fontsize=13, fontweight='bold')
    for i in range(n):
        img = np.array(Image.open(image_paths[i]).convert('RGB'))
        msk = np.array(Image.open(mask_paths[i]).convert('L'))
        axes[0,i].imshow(img);            axes[0,i].set_title(f'Image {i+1}', fontsize=9); axes[0,i].axis('off')
        axes[1,i].imshow(msk, cmap='gray'); axes[1,i].set_title(f'Mask {i+1}',  fontsize=9); axes[1,i].axis('off')
    plt.tight_layout(); plt.show()

# Training data lives flat in TrainDataset/image/ and TrainDataset/masks/
train_imgs = sorted((DATA_ROOT / 'TrainDataset' / 'image').glob('*'))
train_msks = sorted((DATA_ROOT / 'TrainDataset' / 'masks').glob('*'))
show_samples(train_imgs, train_msks, 'Training samples (Kvasir + CVC-ClinicDB mixed)', n=4)

In [ ]:
# Test datasets — each in its own subfolder
test_datasets = {
    'Kvasir (seen test)':          TEST_ROOT / 'Kvasir',
    'CVC-ClinicDB (seen test)':    TEST_ROOT / 'CVC-ClinicDB',
    'CVC-ColonDB (unseen)':        TEST_ROOT / 'CVC-ColonDB',
    'ETIS-LaribPolypDB (unseen)':  TEST_ROOT / 'ETIS-LaribPolypDB',
    'CVC-300 (unseen)':            TEST_ROOT / 'CVC-300',
}
for name, folder in test_datasets.items():
    imgs = sorted((folder / 'images').glob('*'))
    msks = sorted((folder / 'masks').glob('*'))
    show_samples(imgs, msks, name, n=4)

## 6. Augmentation Preview

In [ ]:
from src.data import get_train_transform

transform = get_train_transform(img_size=352)
raw_img = np.array(Image.open(train_imgs[0]).convert('RGB'))
raw_msk = np.array(Image.open(train_msks[0]).convert('L'))
raw_msk_f = (raw_msk > 127).astype(np.float32)

fig, axes = plt.subplots(2, 6, figsize=(18, 6))
fig.suptitle('Same image — 5 random augmentations', fontsize=13)
axes[0,0].imshow(raw_img);              axes[0,0].set_title('Original'); axes[0,0].axis('off')
axes[1,0].imshow(raw_msk, cmap='gray'); axes[1,0].set_title('GT mask');  axes[1,0].axis('off')
for i in range(1, 6):
    out = transform(image=raw_img, mask=raw_msk_f)
    img_t = out['image'].permute(1,2,0).numpy()
    img_d = np.clip(img_t * [0.229,0.224,0.225] + [0.485,0.456,0.406], 0, 1)
    axes[0,i].imshow(img_d);                       axes[0,i].set_title(f'Aug {i}'); axes[0,i].axis('off')
    axes[1,i].imshow(out['mask'].numpy(), cmap='gray'); axes[1,i].set_title(f'Mask {i}'); axes[1,i].axis('off')
plt.tight_layout(); plt.show()

## 7. Evaluation Metrics — Smoke Test

All six PraNet metrics: perfect prediction should score 1.0, all-zeros should score 0.0.

In [ ]:
from src.metrics import dice_score, iou_score, mae_score, weighted_f_measure, s_measure, e_measure

gt  = (np.array(Image.open(train_msks[0]).convert('L')) > 127).astype(np.float32)
rng = np.random.default_rng(0)

cases = {
    'Perfect  (pred = gt)': gt,
    'Random   (uniform)':   rng.uniform(0, 1, gt.shape).astype(np.float32),
    'All zeros':            np.zeros_like(gt),
}

print(f'{"Case":<24} {"Dice":>7} {"IoU":>7} {"MAE":>7} {"wFm":>7} {"Sm":>7} {"Em":>7}')
print('-' * 72)
for name, pred in cases.items():
    b = (pred >= 0.5).astype(np.float32)
    print(f'{name:<24}'
          f' {dice_score(b,gt):>7.4f} {iou_score(b,gt):>7.4f} {mae_score(pred,gt):>7.4f}'
          f' {weighted_f_measure(b,gt):>7.4f} {s_measure(pred,gt):>7.4f} {e_measure(pred,gt):>7.4f}')

In [ ]:
from src.metrics import MetricTracker

tracker = MetricTracker()
for i in range(10):
    gt_i   = (np.array(Image.open(train_msks[i]).convert('L')) > 127).astype(np.float32)
    pred_i = rng.uniform(0, 1, gt_i.shape).astype(np.float32)
    tracker.update(pred_i, gt_i)

print('MetricTracker over 10 images (random predictions):')
print(' ', tracker.summary())

## 8. Polyp Size Distribution

In [ ]:
sizes = [100 * (np.array(Image.open(p).convert('L')) > 127).mean()
         for p in train_msks]

plt.figure(figsize=(8, 4))
plt.hist(sizes, bins=30, edgecolor='black', color='steelblue', alpha=0.8)
plt.xlabel('Polyp area (% of image)'); plt.ylabel('Count')
plt.title('Training set — polyp size distribution (n=1,450)')
plt.axvline(np.mean(sizes), color='red', linestyle='--', label=f'Mean={np.mean(sizes):.1f}%')
plt.legend(); plt.tight_layout(); plt.show()
print(f'Mean={np.mean(sizes):.1f}%  Median={np.median(sizes):.1f}%  Std={np.std(sizes):.1f}%')

## Phase 1 Summary

| Deliverable | Status |
|---|---|
| Repository & dependencies | ✅ |
| All 5 datasets downloaded | ✅ |
| PraNet split builder | ✅ `src/data/dataset.py` |
| Zero train/test overlap | ✅ |
| All 6 evaluation metrics | ✅ `src/metrics/segmentation.py` |
| Augmentation pipeline | ✅ `src/data/transforms.py` |

**Next:** `02_unet_baseline.ipynb` — U-Net training on T4 GPU (~1–2 hrs).